In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)

## Data

In [ ]:
def sample_x0(n, D):
    return torch.randn(n, D)

In [ ]:
def sample_x1(n, weights, means, stds):
    k = torch.multinomial(weights, n, replacement=True)
    return means[k] + stds[k, None] * torch.randn(n, means.shape[1])

1D Mixture gaussian

In [ ]:
weights = torch.tensor([0.5, 0.5])
means = torch.tensor([[1.0], [-1.0]])
stds = torch.tensor([0.3, 0.3])

n = 500
x_1 = sample_x1(n, weights, means, stds)

$$
p(x) = \sum_k w_k \,\mathcal N(x;\,\mu_k, \sigma_k^2),\qquad \mathcal N(x;\,\mu, \sigma^2) = \frac{1}{\sigma\sqrt{2\pi}} \exp\!\left(-\frac12\Big(\frac{x-\mu}{\sigma}\Big)^2\right)
$$

In [ ]:
x = torch.linspace(-4, 4, 1000)
mu = means[:, 0] # (2, 1) -> (2,)
dens = (weights * torch.exp(-0.5 * ((x[:, None] - mu) / stds) ** 2)
        / (stds * (2 * torch.pi) ** 0.5)).sum(dim=1)

plt.hist(x_1[:, 0], bins=60, density=True, alpha=0.5, label="samples")
plt.plot(x, dens, label="true density")
plt.legend()
plt.show()

## 理想經驗場

- 對每個終點 $x_1$ 與給定的當下位置 $x_t$，反推出對應的起點 $x_0$ 與速度 $x_1 - x_0$
- 用起點的機率當權重，再平均各配對的速度

0. 設定：起點 $x_0 \sim \mathcal N(0, I)$，終點 $x_1$ 從訓練集 $\{x_1^{(1)},\dots,x_1^{(N)}\}$ 中均勻抽取

1. 給定當下位置 $x_t$、時間 $t$，對於每個終點 $x^{(i)}_1$，反推起點 $x_0$

$$
\begin{aligned}
x_t &= t\,x^{(i)}_1 + (1-t)\,x_0 \\
\implies x_0 &= \frac{x_t - t\,x^{(i)}_1}{1 - t} 
\end{aligned}
$$



2. 每一對的速度

$$
\begin{aligned}
x_1 - x_0 = x_1 - \frac{x_t - t\,x_1}{1 - t} = \frac{x_1 - x_t}{1 - t} 
\end{aligned}
$$

3. 

In [ ]:
def empirical_v(x, t, data):
    # x: (B, D), data: (N, D)
    sq = ((x[:, None, :] - t * data[None]) ** 2).sum(dim=2) # (B, N)
    w = torch.softmax(-sq / (2 * (1 - t) ** 2), dim=1)
    return (w @ data - x) / (1 - t)

In [ ]:
def gmm_zhat(x, t, weights, means, stds):
    a = (1 - t) ** 2 + t ** 2 * stds ** 2
    logr = torch.log(weights) - 0.5 * torch.log(a) - (x[:, None] - t * means) ** 2 / (2 * a)
    r = torch.softmax(logr, dim=1)
    cond = means + t * stds ** 2 / a * (x[:, None] - t * means)
    return (r * cond).sum(dim=1)

In [ ]:
def velocity(zhat, x, t):
    return (zhat - x) / (1 - t)